In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [2]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


10

In [3]:
# =======================================================================================
#
# BLOCK 3: LOAD ALL PRE-TRAINED MODEL PREDICTIONS
#
# =======================================================================================
PREDS_PATH = './model_predictions/'
print("Loading all base model predictions from saved .npy files...")

# --- Load XGBoost Predictions ---
# In your training script, these were oof_xgb_preds and test_mean_preds
oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy') 
test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
print("  - XGBoost predictions loaded.")

# --- Load CatBoost Predictions ---
# In your training script, these were oof_cb_preds and test_catboost_preds
oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy') # Ensure this matches the saved file name
print("  - CatBoost predictions loaded.")

# --- Load Neural Network Predictions ---
oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
print("  - Neural Network predictions loaded.")
    
y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
print("\nAll predictions loaded successfully.")

# --- OOF RMSE of Loaded Models ---
print("\n--- OOF RMSE of All Base Models ---")
print(f"XGBoost  : ${np.sqrt(mean_squared_error(y_true, oof_xgb_preds)):,.2f}")
print(f"CatBoost : ${np.sqrt(mean_squared_error(y_true, oof_cb_preds)):,.2f}")
print(f"NeuralNet: ${np.sqrt(mean_squared_error(y_true, oof_nn_preds)):,.2f}")

Loading all base model predictions from saved .npy files...
  - XGBoost predictions loaded.
  - CatBoost predictions loaded.
  - Neural Network predictions loaded.

All predictions loaded successfully.

--- OOF RMSE of All Base Models ---
XGBoost  : $98,990.27
CatBoost : $98,246.71
NeuralNet: $108,435.86


In [4]:
# =======================================================================================
#
# BLOCK 4: FIND THE OPTIMAL 3-MODEL BLEND
#
# =======================================================================================
from scipy.optimize import minimize

print("\n--- Searching for the optimal 3-model blend weights ---")

# Stack the OOF predictions into a single array for the optimizer
oof_preds_stack = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T

def get_ensemble_rmse(weights):
    final_prediction = np.dot(oof_preds_stack, weights)
    return np.sqrt(mean_squared_error(y_true, final_prediction))

initial_weights = [1/3, 1/3, 1/3]
constraints = ({'type': 'eq', 'fun': lambda w: 1 - np.sum(w)})
bounds = [(0, 1)] * len(initial_weights)

result = minimize(get_ensemble_rmse, initial_weights, method='SLSQP', bounds=bounds, constraints=constraints)

best_weights = result.x
best_ensemble_rmse = result.fun

print("\n" + "="*50)
print("             OPTIMAL 3-MODEL BLEND SEARCH COMPLETE")
print("="*50)
print(f"Lowest Ensemble RMSE achieved: ${best_ensemble_rmse:,.2f}")
print(f"Optimal Weight for XGBoost  : {best_weights[0]:.4f}")
print(f"Optimal Weight for CatBoost : {best_weights[1]:.4f}")
print(f"Optimal Weight for NeuralNet: {best_weights[2]:.4f}")
print("="*50)


--- Searching for the optimal 3-model blend weights ---

             OPTIMAL 3-MODEL BLEND SEARCH COMPLETE
Lowest Ensemble RMSE achieved: $96,778.10
Optimal Weight for XGBoost  : 0.4100
Optimal Weight for CatBoost : 0.4779
Optimal Weight for NeuralNet: 0.1121


In [5]:
# =======================================================================================
#
# BLOCK 5 (CATBOOST ERROR MODEL EXPERIMENT): THE FULL SUPER-ENSEMBLE PIPELINE
#
# =======================================================================================
import catboost as cb # Make sure CatBoost is imported

# --- PART A: Create the Super-Ensemble Mean (NO CHANGES) ---
# This part is identical, as we are still using the same mean models to create the ensemble.
print("\n--- PART A: Creating the super-ensemble mean predictions with optimal weights ---")
oof_ensemble_mean = (oof_xgb_preds * best_weights[0] + 
                     oof_cb_preds * best_weights[1] + 
                     oof_nn_preds * best_weights[2])
test_ensemble_mean = (test_xgb_preds * best_weights[0] + 
                      test_cb_preds * best_weights[1] + 
                      test_nn_preds * best_weights[2])
print("Super-ensemble predictions created with optimal weights.")


# --- PART B: Tune a New *CatBoost* Error Model for the Super-Ensemble ---
# This entire section is modified to use CatBoost instead of LightGBM.
print("\n--- PART B: Tuning a new CATBOOST error model for the super-ensemble's errors ---")
error_target_ensemble = np.abs(y_true - oof_ensemble_mean)
X_for_error_ensemble = X.copy()
X_for_error_ensemble['mean_pred_oof'] = oof_ensemble_mean

N_OPTUNA_TRIALS = 75 # Using the same number of trials for a fair comparison

# --- CHANGED: Define the objective function specifically for CatBoost ---
def create_catboost_error_objective(X_features, y_error):
    X_train, X_val, y_train, y_val = train_test_split(X_features, y_error, test_size=0.25, random_state=RANDOM_STATE)
    def objective(trial):
        # Using a proven search space for CatBoost
        params = {
            'iterations': 3000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 6, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-4, 10.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'random_strength': trial.suggest_float('random_strength', 1e-4, 10.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_seed': RANDOM_STATE, 'verbose': 0, 'eval_metric': 'RMSE'
        }
        # --- CHANGED: Use cb.CatBoostRegressor ---
        model = cb.CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=0)
        return np.sqrt(mean_squared_error(y_val, model.predict(X_val)))
    return objective

# --- CHANGED: The study now runs the CatBoost objective function ---
study_error_ensemble_cb = optuna.create_study(direction='minimize')
study_error_ensemble_cb.optimize(create_catboost_error_objective(X_for_error_ensemble, error_target_ensemble), n_trials=N_OPTUNA_TRIALS)
best_params_error_ensemble = study_error_ensemble_cb.best_params
# Print metric remains the same
print(f"\nOptimal CatBoost Error Model Params Found (RMSE: ${study_error_ensemble_cb.best_value:,.2f})")

# --- PART C: K-Fold Train the Tuned *CatBoost* Error Model ---
# This section is modified to train with the tuned CatBoost model.
print("\n--- PART C: K-Fold training the new CatBoost error model ---")
final_params_error_ensemble = best_params_error_ensemble.copy()
# --- CHANGED: Update with fixed CatBoost parameters ---
final_params_error_ensemble.update({'iterations': 3000, 'random_seed': RANDOM_STATE, 'verbose': 0, 'eval_metric': 'RMSE'})
oof_error_preds_ensemble = np.zeros(len(X))
test_error_preds_ensemble = np.zeros(len(X_test))
X_test_for_error_ensemble = X_test.copy()
X_test_for_error_ensemble['mean_pred_oof'] = test_ensemble_mean
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_ensemble, grade_for_stratify)):
    print(f"  Training Super-Ensemble Error Model (CatBoost) - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_ensemble.iloc[train_idx], X_for_error_ensemble.iloc[val_idx]
    y_train, y_val = error_target_ensemble.iloc[train_idx], error_target_ensemble.iloc[val_idx]
    # --- CHANGED: Use cb.CatBoostRegressor ---
    model = cb.CatBoostRegressor(**final_params_error_ensemble)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=0)
    oof_error_preds_ensemble[val_idx] = model.predict(X_val)
    test_error_preds_ensemble += model.predict(X_test_for_error_ensemble) / N_SPLITS

# --- PART D: Final Calibration and Showdown (NO CHANGES) ---
# This part is identical. It just takes the new `oof_error_preds_ensemble` generated by the CatBoost
# model and runs them through the same calibration process.
print("\n--- PART D: Calibrating the final interval and getting the score ---")
OLD_BEST_SCORE = 297707.05 # Your current best score from the 2-model blend
oof_error_final_ensemble = np.clip(oof_error_preds_ensemble, 0, None)
best_score_ensemble = float('inf')
best_a_ensemble, best_b_ensemble = 1.0, 1.0
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_ensemble_mean - oof_error_final_ensemble * a
        high = oof_ensemble_mean + oof_error_final_ensemble * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score_ensemble:
            best_score_ensemble = score
            best_a_ensemble, best_b_ensemble = a, b

# Print metrics remain the same
print("\n" + "="*60)
print("             THE SUPER-ENSEMBLE PIPELINE: FINAL SHOWDOWN")
print("="*60)
print(f"Previous Best Score (2-Model Blend) : {OLD_BEST_SCORE:,.2f}")
print(f"New SUPER-ENSEMBLE Pipeline Score (with CatBoost Error Model) : {best_score_ensemble:,.2f}")
print(f"  (Using optimal multipliers a={best_a_ensemble:.2f}, b={best_b_ensemble:.2f})")

if best_score_ensemble < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The CatBoost error model pushed the super-ensemble to a new best score!")
else:
    print("\nCONCLUSION: Close! The pipeline with the XGBoost error model remains superior.")

[I 2025-07-16 22:07:57,157] A new study created in memory with name: no-name-4033b128-b37f-46c5-a9a0-fa3ef2317a84



--- PART A: Creating the super-ensemble mean predictions with optimal weights ---
Super-ensemble predictions created with optimal weights.

--- PART B: Tuning a new CATBOOST error model for the super-ensemble's errors ---


[I 2025-07-16 22:08:00,843] Trial 0 finished with value: 62434.479472283165 and parameters: {'learning_rate': 0.07744902009364653, 'depth': 7, 'l2_leaf_reg': 0.0012488461196505003, 'subsample': 0.7377791359113903, 'random_strength': 4.655818821119464, 'bagging_temperature': 0.8023504119628592}. Best is trial 0 with value: 62434.479472283165.
[I 2025-07-16 22:08:08,033] Trial 1 finished with value: 62195.946316288995 and parameters: {'learning_rate': 0.06580368538680119, 'depth': 9, 'l2_leaf_reg': 0.005870032986944189, 'subsample': 0.948819678367371, 'random_strength': 0.004829308094645858, 'bagging_temperature': 0.23333951061838598}. Best is trial 1 with value: 62195.946316288995.
[I 2025-07-16 22:08:18,817] Trial 2 finished with value: 62085.71731638705 and parameters: {'learning_rate': 0.019509718047209093, 'depth': 7, 'l2_leaf_reg': 0.019238122467236864, 'subsample': 0.5987250746721159, 'random_strength': 0.00014326915336984204, 'bagging_temperature': 0.639587926898705}. Best is tri


Optimal CatBoost Error Model Params Found (RMSE: $61,658.58)

--- PART C: K-Fold training the new CatBoost error model ---
  Training Super-Ensemble Error Model (CatBoost) - Fold 1/5...
  Training Super-Ensemble Error Model (CatBoost) - Fold 2/5...
  Training Super-Ensemble Error Model (CatBoost) - Fold 3/5...
  Training Super-Ensemble Error Model (CatBoost) - Fold 4/5...
  Training Super-Ensemble Error Model (CatBoost) - Fold 5/5...

--- PART D: Calibrating the final interval and getting the score ---

             THE SUPER-ENSEMBLE PIPELINE: FINAL SHOWDOWN
Previous Best Score (2-Model Blend) : 297,707.05
New SUPER-ENSEMBLE Pipeline Score (with CatBoost Error Model) : 297,534.88
  (Using optimal multipliers a=1.95, b=2.21)

CONCLUSION: VICTORY! The CatBoost error model pushed the super-ensemble to a new best score!


In [6]:
# =======================================================================================
#
# BLOCK 8: STAGE 5 - CREATE FINAL SUBMISSION (CORRECTED)
#
# =======================================================================================
OLD_BEST_SCORE = 301999
if best_score_ensemble < OLD_BEST_SCORE:
    print("\n--- Creating the new champion submission file... ---")
    
    # --- THE FIX IS HERE ---
    # Ignore any previous 'test_ids' variable.
    # Load the correct, original IDs directly from the test.csv file.
    # We use `usecols` to make this very fast and memory-efficient.
    print("Loading original IDs from test.csv...")
    correct_test_ids = pd.read_csv(DATA_PATH + 'test.csv', usecols=['id'])['id']
    
    # The rest of the prediction calculations are correct as they are already in the right order.
    test_error_final_ensemble = np.clip(test_error_preds_ensemble, 0, None)
    final_lower = test_ensemble_mean - test_error_final_ensemble * best_a_ensemble
    final_upper = test_ensemble_mean + test_error_final_ensemble * best_b_ensemble
    final_upper = np.maximum(final_lower, final_upper)
    
    # Create the submission DataFrame using the CORRECT IDs.
    submission_df = pd.DataFrame({'id': correct_test_ids, 'pi_lower': final_lower, 'pi_upper': final_upper})
    submission_filename = 'submission_ensemble_pipeline_v1_CORRECT_IDS.csv'
    submission_df.to_csv(submission_filename, index=False)

    print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
    display(submission_df.head())
else:
    print("\nNew submission file not created as the score was not an improvement.")


--- Creating the new champion submission file... ---
Loading original IDs from test.csv...

'submission_ensemble_pipeline_v1_CORRECT_IDS.csv' created successfully! Good luck on the leaderboard!


,id,pi_lower,pi_upper
0,200000,837952.457711,1.053429e+06
1,200001,563855.067253,8.031260e+05
2,200002,444926.481940,6.520687e+05
3,200003,287053.673334,4.202947e+05
4,200004,306813.109837,7.464655e+05
